# Benchmark Example: CSV vs Iceberg (TPC-H Q1)

Notebook ini menjalankan query group-by sederhana (Q1-style) untuk membandingkan CSV vs Iceberg.
Output utama: tabel ringkasan waktu eksekusi dan speedup.

In [ ]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import Row

JARS = ",".join([
    "/home/jovyan/extra-jars/iceberg-spark-runtime-3.5_2.12-1.4.3.jar",
    "/home/jovyan/extra-jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/extra-jars/aws-java-sdk-bundle-1.12.262.jar",
])
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {JARS} pyspark-shell"

spark = (
    SparkSession.builder
    .appName("benchmark-example")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://lakehouse/")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print("Spark version:", spark.version)

In [ ]:
LINEITEM_SCHEMA = (
    "l_orderkey LONG, l_partkey LONG, l_suppkey LONG, l_linenumber INT, "
    "l_quantity DECIMAL(15,2), l_extendedprice DECIMAL(15,2), l_discount DECIMAL(15,2), "
    "l_tax DECIMAL(15,2), l_returnflag STRING, l_linestatus STRING, l_shipdate DATE, "
    "l_commitdate DATE, l_receiptdate DATE, l_shipinstruct STRING, l_shipmode STRING, "
    "l_comment STRING, _trailing STRING"
)

QUERY_SQL = (
    "SELECT l_returnflag, l_linestatus, "
    "SUM(l_quantity) AS sum_qty, SUM(l_extendedprice) AS sum_base_price, "
    "AVG(l_discount) AS avg_disc, COUNT(*) AS count_order "
    "FROM {table} "
    "GROUP BY l_returnflag, l_linestatus "
    "ORDER BY l_returnflag, l_linestatus"
)

def timed_runs(sql_text, warmup, runs):
    for _ in range(warmup):
        spark.sql(sql_text).count()
    times = []
    row_count = None
    for _ in range(runs):
        spark.catalog.clearCache()
        start = time.perf_counter()
        count = spark.sql(sql_text).count()
        times.append(time.perf_counter() - start)
        if row_count is None:
            row_count = count
    return times, row_count

warmup_runs = 1
runs = 3

In [ ]:
# CSV view
csv_df = (
    spark.read
    .option("delimiter", "|")
    .option("header", "false")
    .schema(LINEITEM_SCHEMA)
    .csv("s3a://tpch-raw/lineitem.tbl")
    .drop("_trailing")
)
csv_df.createOrReplaceTempView("csv_lineitem")

csv_times, csv_rows = timed_runs(QUERY_SQL.format(table="csv_lineitem"), warmup_runs, runs)
ice_times, ice_rows = timed_runs(QUERY_SQL.format(table="local.tpch.lineitem"), warmup_runs, runs)

csv_avg = sum(csv_times) / len(csv_times)
ice_avg = sum(ice_times) / len(ice_times)
speedup = csv_avg / ice_avg if ice_avg > 0 else None

results_df = spark.createDataFrame([
    Row(source="CSV", avg_sec=csv_avg, rows=csv_rows),
    Row(source="Iceberg", avg_sec=ice_avg, rows=ice_rows),
])
results_df.show(truncate=False)
print(f"Speedup (CSV / Iceberg): {speedup:.2f}x")

In [ ]:
# Optional plot
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed. Run: pip install matplotlib")
else:
    labels = ["CSV", "Iceberg"]
    values = [csv_avg, ice_avg]
    plt.bar(labels, values)
    plt.ylabel("Average time (sec)")
    plt.title("CSV vs Iceberg - Q1")
    plt.show()